**CineLexi: Corpus-Based Film Review Vocabulary Resource (German–Russian)**

**Research question:** What lexical profile characterises the film review genre in German and Russian, and what are the cross-lingual asymmetries in this profile?

Corpus: 900 user reviews — 15 films × 5 genres (3 per genre) × 30 reviews × 2 languages.

Sources: Kinopoisk.ru (RU) + Filmstarts.de (DE).

**Section 1: Setup**

Install all dependencies and import libraries. Selenium + headless Chrome is required for Filmstarts (site renders reviews via JavaScript). spaCy md models provide accurate lemmatisation and word vectors. BeautifulSoup is used for scraping GermanFluent. scikit-learn provides TF-IDF for Filter A.

In [1]:
!pip install selenium webdriver-manager spacy wordfreq openpyxl beautifulsoup4 scikit-learn -q
!apt-get update -qq
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -y --fix-missing ./google-chrome-stable_current_amd64.deb -q
!python -m spacy download de_core_news_md -q
!python -m spacy download ru_core_news_md -q

import io, re, string, zipfile, time, difflib
import requests
import pandas as pd
from collections import Counter, defaultdict
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
from google.colab import files

import spacy
from wordfreq import zipf_frequency
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

print("Setup complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.8/511.8 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/135.7 kB 9.8 MB/s eta 0:00:00
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  at-spi2-common at-spi2-core gsettings-desktop-schemas libatk-bridge2.0-0t64
  libatk1.0-0t64 libatspi2.0-0t64 libxcomposite1 libxtst6 session-migration
The following NEW packages will be installed:

Installs all dependencies; installs Chrome for Selenium; downloads German and Russian spaCy md models.

**Film metadata** — 15 films, 5 genres × 3 per genre.

All films are from the IMDb Top 250 and present on both Kinopoisk and Filmstarts. kp_id is the Kinopoisk internal ID; fs_id is the Filmstarts internal ID.

In [2]:
FILMS = [
    # DRAMA
    {'kp_id': 326,    'name_en': 'The Shawshank Redemption', 'name_ru': 'Побег из Шоушенка',     'year': 1994, 'genre': 'drama',    'fs_id': '11736'},
    {'kp_id': 435,    'name_en': 'The Green Mile',           'name_ru': 'Зелёная миля',          'year': 1999, 'genre': 'drama',    'fs_id': '22779'},
    {'kp_id': 361,    'name_en': 'Fight Club',               'name_ru': 'Бойцовский клуб',       'year': 1999, 'genre': 'drama',    'fs_id': '21189'},
    # SCI-FI
    {'kp_id': 258687, 'name_en': 'Interstellar',             'name_ru': 'Интерстеллар',          'year': 2014, 'genre': 'sci-fi',   'fs_id': '114782'},
    {'kp_id': 447301, 'name_en': 'Inception',                'name_ru': 'Начало',                'year': 2010, 'genre': 'sci-fi',   'fs_id': '143692'},
    {'kp_id': 301,    'name_en': 'The Matrix',               'name_ru': 'Матрица',               'year': 1999, 'genre': 'sci-fi',   'fs_id': '19776'},
    # CRIME / THRILLER
    {'kp_id': 342,    'name_en': 'Pulp Fiction',             'name_ru': 'Криминальное чтиво',    'year': 1994, 'genre': 'crime',    'fs_id': '10126'},
    {'kp_id': 345,    'name_en': 'The Silence of the Lambs', 'name_ru': 'Молчание ягнят',        'year': 1991, 'genre': 'thriller', 'fs_id': '6641'},
    {'kp_id': 195434, 'name_en': 'No Country for Old Men',   'name_ru': 'Старикам тут не место', 'year': 2007, 'genre': 'crime',    'fs_id': '110096'},
    # ACTION
    {'kp_id': 474,    'name_en': 'Gladiator',                'name_ru': 'Гладиатор',             'year': 2000, 'genre': 'action',   'fs_id': '24944'},
    {'kp_id': 389,    'name_en': 'Leon',                     'name_ru': 'Леон',                  'year': 1994, 'genre': 'action',   'fs_id': '9684'},
    {'kp_id': 111543, 'name_en': 'The Dark Knight',          'name_ru': 'Тёмный рыцарь',         'year': 2008, 'genre': 'action',   'fs_id': '115362'},
    # COMEDY / ROMANCE
    {'kp_id': 341,    'name_en': 'Amelie',                   'name_ru': 'Амели',                 'year': 2001, 'genre': 'romance',  'fs_id': '27063'},
    {'kp_id': 683999, 'name_en': 'The Grand Budapest Hotel', 'name_ru': 'Отель Гранд Будапешт',  'year': 2014, 'genre': 'comedy',   'fs_id': '207825'},
    {'kp_id': 448,    'name_en': 'Forrest Gump',             'name_ru': 'Форрест Гамп',          'year': 1994, 'genre': 'comedy',   'fs_id': '10568'},
]
TARGET_PER_FILM = 30
N_FILMS = len(FILMS)
print(f"Corpus: {N_FILMS} films × {TARGET_PER_FILM} reviews × 2 languages = "
      f"{N_FILMS*TARGET_PER_FILM*2} reviews")

Corpus: 15 films × 30 reviews × 2 languages = 900 reviews


15 films, 5 genres, 30 reviews per film per language → 900 total reviews expected.

**Section 2: Russian Data Collection — Kinopoisk API**

Unofficial Kinopoisk API (poiskkino.dev) returns paginated JSON. Only reviews longer than 200 characters are collected — shorter texts lack sufficient linguistic content.

In [3]:
KP_KEY = 'BD6360M-E0P4HRS-P846AZM-90P5JGM'
rows_ru = []

for film in FILMS:
    collected, page = [], 1
    while len(collected) < TARGET_PER_FILM:
        r = requests.get(
            'https://api.poiskkino.dev/v1.4/review',
            headers={'X-API-KEY': KP_KEY},
            params={'movieId': film['kp_id'], 'limit': 10, 'page': page}
        )
        docs = r.json().get('docs', [])
        if not docs: break
        for rev in docs:
            text = rev.get('review') or ''
            if len(text) > 200:
                collected.append({**film, 'review_ru': text, 'length_ru': len(text)})
                if len(collected) >= TARGET_PER_FILM: break
        page += 1
        time.sleep(0.5)
    rows_ru.extend(collected)
    print(f"  {film['name_en']}: {len(collected)} reviews")

df_ru = pd.DataFrame(rows_ru)
df_ru['review_idx'] = df_ru.groupby('name_en').cumcount() + 1
df_ru.to_csv('kinopoisk_reviews.csv', index=False, encoding='utf-8-sig')
print(f"\nTotal: {len(df_ru)} RU reviews saved.")

  The Shawshank Redemption: 30 reviews
  The Green Mile: 30 reviews
  Fight Club: 30 reviews
  Interstellar: 30 reviews
  Inception: 30 reviews
  The Matrix: 30 reviews
  Pulp Fiction: 30 reviews
  The Silence of the Lambs: 30 reviews
  No Country for Old Men: 30 reviews
  Gladiator: 30 reviews
  Leon: 30 reviews
  The Dark Knight: 30 reviews
  Amelie: 30 reviews
  The Grand Budapest Hotel: 30 reviews
  Forrest Gump: 30 reviews

Total: 450 RU reviews saved.


In [4]:
# preview: one review per film (Russian subcorpus)
# median-length review per film for representativeness.
from IPython.display import display, HTML

def median_review(df, text_col):
    return (df.sort_values(text_col)
              .groupby('name_en', group_keys=False)
              .apply(lambda g: g.iloc[len(g)//2]))

sample_ru = (median_review(df_ru, 'length_ru')
             [['name_en','year','genre','length_ru','review_ru']]
             .copy())
sample_ru['review_ru'] = sample_ru['review_ru'].str[:200] + '...'
sample_ru = sample_ru.reset_index(drop=True)
sample_ru.columns = ['Film','Year','Genre','Length','Review (first 200 chars)']

styled_ru = (sample_ru.style
    .set_properties(**{
        'text-align': 'left',
        'white-space': 'normal',
        'word-wrap': 'break-word',
        'max-width': '600px',
        'font-size': '12px',
        'vertical-align': 'top',
    })
    .set_table_styles([
        {'selector': 'th', 'props': [
            ('background-color', '#f0f0f0'),
            ('font-weight', 'bold'),
            ('text-align', 'left'),
            ('padding', '6px'),
            ('font-size', '12px'),
        ]},
        {'selector': 'td', 'props': [
            ('padding', '6px'),
            ('border-bottom', '1px solid #e0e0e0'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
        ]},
    ])
    .hide(axis='index')
)

print(f"Russian subcorpus — one review per film ({len(sample_ru)} films)\n")
display(styled_ru)

Russian subcorpus — one review per film (15 films)



/tmp/ipykernel_656/4164059822.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.iloc[len(g)//2]))


Film,Year,Genre,Length,Review (first 200 chars)
Amelie,2001,romance,2231,"Знаете ли вы, что все события, происходящие в нашем мире, даже самые незначительные, взаимосвязаны самым удивительным и чудесным образом? Как полет крошечной мухи может вызвать где-то далеко мощный ур..."
Fight Club,1999,drama,2860,"Общее впечатление: Шедевральная лента Дэвида Финчера, занимающая 9 место в списке 250 лучших фильмов по версии сайта КиноПоиск. Уже знаете 1 правило «Бойцовского клуба»? Не упоминать о «Бойцовском клу..."
Forrest Gump,1994,comedy,2183,"Честно говоря, долго колебался, ставить этому фильму 6 или 7. Обычно на 6 я оцениваю фильмы, которые в целом не представляют из себя ничего особенного, но отдельные моменты в них мне понравились и пок..."
Gladiator,2000,action,2450,"«Гладиатор» — тот случай, когда на экране очень хорошо видно колоссальную работу всей съёмочной команды. Костюмы, операторская работа, музыка, постановка боёв, массовые сцены — всё собрано так, чтобы ..."
Inception,2010,sci-fi,3042,"После нашумевшего Оскара 2024 года, где заветные статуэтки забрали себе режиссер Кристофер Нолан и актер Киллиан Мерфи, я просто не мог не ознакомиться с другой их совместной работой. Хоть фильм яв..."
Interstellar,2014,sci-fi,2605,"Киношедевр «Интерстеллар», выпущенный в 2014 году и срежиссированный талантливым Кристофером Ноланом, стал не только ярким примером научно-фантастического кино, но и глубоким философским исследованием..."
Leon,1994,action,1968,"Сразу скажу: когда смотрел впервые этот фильм, Портман мне показалась совершенно невзрачной, а Рено - слишком вычурным, а криминальные элементы слишком пугающими. Изменились ли герои в будущем? Как лю..."
No Country for Old Men,2007,crime,2911,"Со времен «Фарго» в коэновской фильмографии не было фильма столь кровавого и жестокого, как «Старикам тут не место», это некий пессимистический апогей их совместного творчества. Может, свою роль сыгра..."
Pulp Fiction,1994,crime,2327,"Успех фильма «Криминальное чтиво», по моему мнению, заключён в том, что эта лента вобрала в себя все цвета и оттенки настоящего классического американского кинематографа. Действительно, до 1994 года н..."
The Dark Knight,2008,action,2589,"Хвалить Хита Леджера в роли Джокера глупо — это сделали все, кто видел фильм. Но и я скажу: это лучший Джокер. Не потому что страшный. Не потому что смешной. Не потому что безумный — безумие в кино эт..."


df_ru and kinopoisk_reviews.csv with up to 450 Russian reviews.

**Section 3: German Data Collection — Filmstarts (Selenium)**

Filmstarts renders reviews via JavaScript — headless Chrome is required. CSS selector: .content-txt.review-card-content. URL pattern: filmstarts.de/kritiken/{fs_id}/userkritiken/?page={n}.

In [5]:
def make_driver():
    options = Options()
    options.add_argument('--headless=new')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.binary_location = '/usr/bin/google-chrome'
    service = Service(ChromeDriverManager().install())
    return webdriver.Chrome(service=service, options=options)

def get_filmstarts_reviews(driver, fs_id, target=30):
    reviews, page = [], 1
    while len(reviews) < target:
        url = f'https://www.filmstarts.de/kritiken/{fs_id}/userkritiken/?page={page}'
        driver.get(url)
        time.sleep(4)
        body = driver.find_element(By.TAG_NAME, 'body').text
        if len(body) < 500 or 'with your agreement' in body.lower(): break
        if '404' in body[:100] or 'FEHLER' in body[:100]: break
        elements = driver.find_elements(By.CSS_SELECTOR, '.content-txt.review-card-content')
        found = 0
        for el in elements:
            txt = el.text.strip()
            if len(txt) > 50:
                reviews.append(txt); found += 1
                if len(reviews) >= target: break
        print(f'    Page {page}: {found} (total: {len(reviews)})')
        if found == 0: break
        page += 1; time.sleep(1)
    return reviews[:target]

print('Starting browser...')
driver = make_driver()
rows_de = []
try:
    for film in FILMS:
        print(f"[{film['name_en']}]")
        reviews = get_filmstarts_reviews(driver, film['fs_id'])
        for i, text in enumerate(reviews):
            rows_de.append({**film, 'review_idx': i+1,
                            'review_de': text, 'length_de': len(text)})
        print(f"  → {len(reviews)} reviews\n")
        time.sleep(2)
finally:
    driver.quit()

df_de = pd.DataFrame(rows_de)
df_de.to_csv('filmstarts_user_reviews.csv', index=False, encoding='utf-8-sig')
print(f"Total: {len(df_de)} DE reviews saved.")

Starting browser...
[The Shawshank Redemption]
    Page 1: 10 (total: 10)
    Page 2: 10 (total: 20)
    Page 3: 10 (total: 30)
  → 30 reviews

[The Green Mile]
    Page 1: 10 (total: 10)
    Page 2: 10 (total: 20)
    Page 3: 9 (total: 29)
    Page 4: 1 (total: 30)
  → 30 reviews

[Fight Club]
    Page 1: 10 (total: 10)
    Page 2: 9 (total: 19)
    Page 3: 10 (total: 29)
    Page 4: 1 (total: 30)
  → 30 reviews

[Interstellar]
    Page 1: 10 (total: 10)
    Page 2: 10 (total: 20)
    Page 3: 10 (total: 30)
  → 30 reviews

[Inception]
    Page 1: 9 (total: 9)
    Page 2: 10 (total: 19)
    Page 3: 10 (total: 29)
    Page 4: 1 (total: 30)
  → 30 reviews

[The Matrix]
    Page 1: 10 (total: 10)
    Page 2: 9 (total: 19)
    Page 3: 10 (total: 29)
    Page 4: 1 (total: 30)
  → 30 reviews

[Pulp Fiction]
    Page 1: 10 (total: 10)
    Page 2: 10 (total: 20)
    Page 3: 10 (total: 30)
  → 30 reviews

[The Silence of the Lambs]
    Page 1: 10 (total: 10)
    Page 2: 10 (total: 20)
    Page 

In [6]:
# preview: one review per film (German subcorpus)
# median-length review per film for representativeness.
from IPython.display import display, HTML

def median_review(df, text_col):
    return (df.sort_values(text_col)
              .groupby('name_en', group_keys=False)
              .apply(lambda g: g.iloc[len(g)//2]))

sample_de = (median_review(df_de, 'length_de')
             [['name_en','year','genre','length_de','review_de']]
             .copy())
sample_de['review_de'] = sample_de['review_de'].str[:200] + '...'
sample_de = sample_de.reset_index(drop=True)
sample_de.columns = ['Film','Year','Genre','Length','Review (first 200 chars)']

styled_de = (sample_de.style
    .set_properties(**{
        'text-align': 'left',
        'white-space': 'normal',
        'word-wrap': 'break-word',
        'max-width': '600px',
        'font-size': '12px',
        'vertical-align': 'top',
    })
    .set_table_styles([
        {'selector': 'th', 'props': [
            ('background-color', '#f0f0f0'),
            ('font-weight', 'bold'),
            ('text-align', 'left'),
            ('padding', '6px'),
            ('font-size', '12px'),
        ]},
        {'selector': 'td', 'props': [
            ('padding', '6px'),
            ('border-bottom', '1px solid #e0e0e0'),
        ]},
        {'selector': 'table', 'props': [
            ('border-collapse', 'collapse'),
            ('width', '100%'),
        ]},
    ])
    .hide(axis='index')
)

print(f"German subcorpus — one review per film ({len(sample_de)} films)\n")
display(styled_de)

German subcorpus — one review per film (15 films)



/tmp/ipykernel_656/615566147.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.iloc[len(g)//2]))


Film,Year,Genre,Length,Review (first 200 chars)
Amelie,2001,romance,424,"Der Film hat viele tolle Bilder, sympathische Darsteller und auch die Story ist mal etwas anderes. Der Anfang hat mir sehr gut gefallen, doch nach ~ 30 Minuten hat es leider angefangen mich zu langwei..."
Fight Club,1999,drama,330,"Fazit: Ein Film, den ich mir öfters anschauen kann, obwohl mir schon jedes kleine Detail bekannt ist. Das liegt an der genialen Inszenierung eines David Finchers. Die Kamerafahrten sind auch heute noc..."
Forrest Gump,1994,comedy,502,Forrest Gump ist auch nach dem vierten oder fünften Mal anschauen ein wunderbar herzlicher Film. Der Film ist perfekt gegliedert und hebt sich vom Mainstream anderer Filme ab. Komplett unverständlich ...
Gladiator,2000,action,368,"Ein auf Schauwerten basiertes, in antikes Setting verkleidetes Kitsch-Märchen auf dem Niveau von Sonntagvormittags-Fernsehen. Der Film überspannt an der Oberfläche alle Charaktere und ihre Konflikte, ..."
Inception,2010,sci-fi,601,"Dieser ganze Jubel um den Film ist mir zuviel. Da werden Erwartungen geweckt, die - was mich betrifft - dann nur enttäuscht werden. Die Story ist elendig verdechselt und so böse unlogisch, dass es ei..."
Interstellar,2014,sci-fi,942,"Was soll man groß zu Nolan sagen, ich habe viel erwartet und der Film hat genau meinen Erwartungen entsprochen. Wer das Weltall und generell das Universum liebt wird hier wirklich seinen Gefallen find..."
Leon,1994,action,346,"Dafür, dass der Film im Jahre 1995 gedreht wurde, muss ich sagen, das das ein richtiges Meisterwekr ist. Die Story ist Einfach nur Toll. Und Jean Reno spielt die rolle des Auftragskillers einfach nur ..."
No Country for Old Men,2007,crime,408,"An diesem Film ist vielleicht etwas künstlerisches, das mag ja sein, aber die Story ist dermaßend abstoßend, willkürlich und unmenschlich, dass für mich damals der Kinobesuch zur Qual wurde. Umso mehr..."
Pulp Fiction,1994,crime,389,"""Pulp Fiction"" ist Tarantino-Kino pur. Schräge Typen, grandiose Dialoge und hier gibt es (wie in ""Sin City"") 3 Kurzgeschichten / Episoden und alle Handlungsstränge kommen am Ende zusammen. Es gibt ein..."
The Dark Knight,2008,action,670,"Erst fragte ich mich ja, ob der ganze Hype um den Film gerechtfertigt ist. Dann hats mich erwischt, Trailer hier, Filmkritik da... ich wollte ihn einfach nurnoch sehen. Karten für die Vorpremiere ..."


df_de and filmstarts_user_reviews.csv with up to 450 German reviews.

**Section 4: Corpus Statistics**

Russian reviews are ~3.4× longer than German on average. This structural imbalance requires normalisation: all frequency data is reported per 1,000 tokens.

In [7]:
total_tokens_de = df_de['review_de'].fillna('').str.split().str.len().sum()
total_tokens_ru = df_ru['review_ru'].fillna('').str.split().str.len().sum()

print("Corpus Statistics \n")
print(f"Reviews: {len(df_de)} DE | {len(df_ru)} RU | {len(df_de)+len(df_ru)} total")
print(f"Tokens:  {total_tokens_de:,} DE | {total_tokens_ru:,} RU")
print(f"Avg length: {df_de['length_de'].mean():.0f} chars (DE) | "
      f"{df_ru['length_ru'].mean():.0f} chars (RU)")
print(f"Length ratio RU/DE: {df_ru['length_ru'].mean()/df_de['length_de'].mean():.1f}x\n")

ru_s = df_ru.groupby('name_en')['length_ru'].agg(['count','mean']).round(0).astype(int)
de_s = df_de.groupby('name_en')['length_de'].agg(['count','mean']).round(0).astype(int)
ru_s.columns = ['RU','avg(RU)']; de_s.columns = ['DE','avg(DE)']
print(ru_s.join(de_s).to_string())

def norm_freq(count, total):
    return round(count / total * 1000, 2)

Corpus Statistics 

Reviews: 450 DE | 450 RU | 900 total
Tokens:  56,098 DE | 182,366 RU
Avg length: 824 chars (DE) | 2769 chars (RU)
Length ratio RU/DE: 3.4x

                          RU  avg(RU)  DE  avg(DE)
name_en                                           
Amelie                    30     2421  30      872
Fight Club                30     3113  30      575
Forrest Gump              30     2733  30     1011
Gladiator                 30     2747  30      827
Inception                 30     3026  30      836
Interstellar              30     2878  30     1635
Leon                      30     2406  30      567
No Country for Old Men    30     3216  30      681
Pulp Fiction              30     2633  30      587
The Dark Knight           30     2611  30     1258
The Grand Budapest Hotel  30     2308  30      878
The Green Mile            30     3076  30      513
The Matrix                30     2917  30      949
The Shawshank Redemption  30     2536  30      621
The Silence of the Lambs

Length imbalance justifies normalisation. norm_freq defined and reused throughout.

**Section 5: CEFR Reference Lists**

Three sources are loaded:

*   5a. GermanFluent (DE, A1–C2) — scraped from germanfluent.com/vocabulary/, licensed CC BY-SA 4.0
*   5b. Goethe-Institut (DE, A1–B1) — official open lists from GitHub
*   5c. Kelly RU (RU, A1–C2) — corpus-based word list, uploaded manually

For German, priority is GermanFluent → Goethe → wordfreq. For Russian, priority is Kelly → wordfreq.

**5a. GermanFluent (DE, A1–C2)**

GermanFluent is a community-curated resource licensed CC BY-SA 4.0, with word data from Wiktionary, Tatoeba and OpenSubtitles. Unlike Goethe, it covers all six CEFR levels for German — including B2 and C1. URL pattern: page 1 is /vocabulary/{level}/; subsequent pages are /vocabulary/{level}/page/{n}/.

In [8]:
LEVELS   = ['a1','a2','b1','b2','c1','c2']
GF_BASE  = 'https://germanfluent.com/vocabulary'
ARTICLES = {'der','die','das'}

def scrape_gf_level(level, max_pages=60):
    words = set()
    for page in range(1, max_pages+1):
        url = (f'{GF_BASE}/{level}/' if page == 1
               else f'{GF_BASE}/{level}/page/{page}/')
        try:
            r = requests.get(url, timeout=15,
                             headers={'User-Agent': 'CineLexi research (NLP4CALL 2026)'})
            if r.status_code != 200: break
            soup  = BeautifulSoup(r.text, 'html.parser')
            spans = (soup.select('aside nav ol li a span.wi-w') or
                     soup.select('span.wi-w'))
            found = set()
            for span in spans:
                tokens  = span.get_text(separator=' ', strip=True).split()
                content = [t for t in tokens if t.lower() not in ARTICLES]
                if not content: continue
                lemma = content[0].strip('.,;:()[]').lower()
                if lemma and lemma.isalpha() and len(lemma) > 1:
                    found.add(lemma)
            if not found: break
            words.update(found)
            time.sleep(0.7)
        except Exception as e:
            print(f'    Error p{page}: {e}'); break
    return words

print('Downloading GermanFluent (CC BY-SA 4.0)...')
germanfluent = {}
for level in LEVELS:
    words = scrape_gf_level(level)
    for w in words:
        if w not in germanfluent:
            germanfluent[w] = level.upper()
    print(f'  {level.upper()}: {len(words)} words')
print(f'Total: {len(germanfluent)} German lemmas with CEFR\n')

  A1: 600 words
  A2: 784 words
  B1: 1604 words
  B2: 2058 words
  C1: 4449 words
  C2: 5450 words
Total: 14931 German lemmas with CEFR



In [9]:
# sample words per level — 8 examples each
import random
random.seed(42)  # for reproducibility

rows = []
for level in ['A1','A2','B1','B2','C1','C2']:
    words = sorted([w for w, lvl in germanfluent.items() if lvl == level])
    sample = random.sample(words, min(8, len(words)))
    rows.append({'Level': level, 'Example words': ', '.join(sample)})

samples = pd.DataFrame(rows)
display(HTML(samples.to_html(index=False, escape=False)))

Level,Example words
A1,"deutsch, ankommen, kamera, hochzeit, gruß, einmal, daran, werden"
A2,"bild, streiten, nutzer, ausgang, ausbildung, blut, gabel, general"
B1,"reaktion, strafe, anlage, senkrecht, erhöhen, wahnsinnig, unglück, visum"
B2,"umgerechnet, index, versorgung, luxemburg, abwarten, flieger, unfair, rebell"
C1,"landstreicher, exzellent, herzanfall, permanent, darsteller, bundesstraße, satan, challenge"
C2,"mixer, matchball, vorhersagen, hämmern, aufgefangen, schlapp, triade, dienstmarke"


germanfluent dict covering German vocabulary across A1–C2.

**5b. Goethe-Institut (DE, A1–B1)**

Official Goethe vocabulary for the Zertifikat exams. Used as a secondary cross-check — GermanFluent takes priority.

In [10]:
BASE_GH = ('https://raw.githubusercontent.com/'
           'ilkermeliksitki/goethe-institute-wordlist/main/')

def fetch_goethe_level(level):
    words = set()
    urls = ([f'{BASE_GH}a1/a1.tsv'] if level == 'a1' else
            [f'{BASE_GH}{level}/{l}.tsv' for l in string.ascii_lowercase])
    for url in urls:
        try:
            r = requests.get(url, timeout=10)
            if r.status_code != 200: continue
            for line in r.text.splitlines():
                if not line.strip() or line.startswith(('Lemma','Word')): continue
                raw   = line.split('\t')[0].strip()
                lemma = raw.split(',')[0].split('(')[0].strip().lower().split()[-1]
                if lemma and lemma.isalpha(): words.add(lemma)
        except Exception: continue
    return words

print('Downloading Goethe-Institut word lists...')
goethe_a1 = fetch_goethe_level('a1')
goethe_a2 = fetch_goethe_level('a2')
goethe_b1 = fetch_goethe_level('b1')
print(f'  A1: {len(goethe_a1)} | A2: {len(goethe_a2)} | B1: {len(goethe_b1)}\n')

  A1: 23 | A2: 1002 | B1: 1720



In [11]:
# preview: Goethe-Institut parsing results
from IPython.display import display, HTML
import pandas as pd
import random
random.seed(42)

# wrap the three sets into a dict for convenience
goethe = {'A1': goethe_a1, 'A2': goethe_a2, 'B1': goethe_b1}

# summary per level
summary = pd.DataFrame({
    'Level': list(goethe.keys()),
    'Words scraped': [len(goethe[l]) for l in goethe],
    'Share (%)': [round(len(goethe[l]) / sum(len(goethe[k]) for k in goethe) * 100, 1)
                  for l in goethe],
})

print(f"Goethe-Institut — {sum(len(s) for s in goethe.values())} lemmas across 3 CEFR levels\n")

print(" Distribution by level")
display(HTML(summary.to_html(index=False, escape=False)))

# sample words per level — 8 examples each
rows = []
for level in ['A1','A2','B1']:
    words = sorted(goethe[level])
    sample = random.sample(words, min(8, len(words)))
    rows.append({'Level': level, 'Example words': ', '.join(sample)})

samples = pd.DataFrame(rows)
print("\n Sample words per level")
display(HTML(samples.to_html(index=False, escape=False)))

# cross-check against GermanFluent
cross = []
for level in ['A1','A2','B1']:
    in_gf = [w for w in goethe[level] if w in germanfluent]
    cross.append({
        'Level': level,
        'Goethe words': len(goethe[level]),
        'Also in GermanFluent': len(in_gf),
        'Overlap (%)': round(len(in_gf) / max(len(goethe[level]), 1) * 100, 1),
    })
cross_df = pd.DataFrame(cross)
print("\n Overlap with GermanFluent")
display(HTML(cross_df.to_html(index=False, escape=False)))

Goethe-Institut — 2745 lemmas across 3 CEFR levels

 Distribution by level


Level,Words scraped,Share (%)
A1,23,0.8
A2,1002,36.5
B1,1720,62.7



 Sample words per level


Level,Example words
A1,"ankreuzen, abfahrt, ab, adresse, achtung, anklicken, abgeben, ankunft"
A2,"renovieren, schuh, wagen, moment, balkon, nur, kein, ansehen"
B1,"an, befinden, dreck, egal, instrument, menge, alter, leistung"



 Overlap with GermanFluent


Level,Goethe words,Also in GermanFluent,Overlap (%)
A1,23,23,100.0
A2,1002,979,97.7
B1,1720,1663,96.7


three sets goethe_a1, goethe_a2, goethe_b1.

**5c. Kelly RU (RU, A1–C2)**

Corpus-based frequency word list for Russian as a foreign language (Kilgarriff et al., 2014). Covers A1–C2. Only nouns (n) and adjectives (adj) are loaded. Download ru_m3.xls from spraakbanken.gu.se/en/resources/kelly before running the next cell.

In [12]:
print('Upload Kelly RU (ru_m3.xls)')
up_kelly = files.upload()
kelly_ru = {}
for fname, data in up_kelly.items():
    df_k      = pd.read_excel(io.BytesIO(data))
    col_lemma = df_k.columns[0]
    col_cefr  = next((c for c in df_k.columns
                      if str(c).upper() in ('CEFR','CEFRLEVEL','LEVEL')), None)
    if col_cefr is None:
        for c in df_k.columns:
            if df_k[c].dropna().astype(str).str.upper().str.match(
                    r'^[ABC][12]$').mean() > 0.3:
                col_cefr = c; break
    col_pos    = next((c for c in df_k.columns
                       if str(c).upper() in ('POS','WORDCLASS','WORD CLASS')), None)
    KEEP_POS   = {'n','adj','noun','adjective'}
    cefr_order = ['A1','A2','B1','B2','C1','C2']
    for _, row in df_k.iterrows():
        lemma = str(row[col_lemma]).strip().lower()
        level = str(row[col_cefr]).strip().upper() if col_cefr else ''
        if level not in set(cefr_order): continue
        if col_pos and str(row[col_pos]).strip().lower() not in KEEP_POS: continue
        if (lemma not in kelly_ru or
                cefr_order.index(level) < cefr_order.index(kelly_ru[lemma])):
            kelly_ru[lemma] = level
    print(f'  Kelly RU: {len(kelly_ru)} entries')

def assign_cefr_de(lemma):
    if lemma in germanfluent:   return germanfluent[lemma], 'GermanFluent'
    if lemma in goethe_a1:      return 'A1', 'Goethe'
    if lemma in goethe_a2:      return 'A2', 'Goethe'
    if lemma in goethe_b1:      return 'B1', 'Goethe'
    z = zipf_frequency(lemma, 'de')
    if z >= 4.0:   return 'B2', 'wordfreq'
    elif z >= 3.0: return 'C1', 'wordfreq'
    elif z > 0:    return 'C2', 'wordfreq'
    return None, None

def assign_cefr_ru(lemma):
    if lemma in kelly_ru: return kelly_ru[lemma], 'Kelly'
    z = zipf_frequency(lemma, 'ru')
    if z >= 4.5:   return 'B1', 'wordfreq'
    elif z >= 4.0: return 'B2', 'wordfreq'
    elif z >= 3.5: return 'C1', 'wordfreq'
    else:          return 'C2', 'wordfreq'

Upload Kelly RU (ru_m3.xls)


Saving ru_m3.xls to ru_m3.xls
  Kelly RU: 5925 entries


In [13]:
# preview: Kelly RU parsing results
from IPython.display import display, HTML
import pandas as pd
import random
random.seed(42)

# summary per CEFR level
kelly_counts = (pd.Series(kelly_ru)
                  .value_counts()
                  .reindex(['A1','A2','B1','B2','C1','C2'])
                  .fillna(0)
                  .astype(int))

summary = pd.DataFrame({
    'Level': kelly_counts.index,
    'Words scraped': kelly_counts.values,
    'Share (%)': (kelly_counts.values / kelly_counts.sum() * 100).round(1),
})

print(f"Kelly RU — {len(kelly_ru)} Russian lemmas across 6 CEFR levels\n")

print("Distribution by level")
display(HTML(summary.to_html(index=False, escape=False)))

# sample words per level — 8 examples each
rows = []
for level in ['A1','A2','B1','B2','C1','C2']:
    words = sorted([w for w, lvl in kelly_ru.items() if lvl == level])
    sample = random.sample(words, min(8, len(words)))
    rows.append({'Level': level, 'Example words': ', '.join(sample)})

samples = pd.DataFrame(rows)
print("\n Sample words per level")
display(HTML(samples.to_html(index=False, escape=False)))

Kelly RU — 5925 Russian lemmas across 6 CEFR levels

Distribution by level


Level,Words scraped,Share (%)
A1,581,9.8
A2,582,9.8
B1,1064,18.0
B2,2374,40.1
C1,954,16.1
C2,370,6.2



 Sample words per level


Level,Example words
A1,"дата, банан, настоящее, медленный, левый, европеец, господин, чеснок"
A2,"гриль, рюкзак, булочка, брошюра, далекий, лестница, медицина, угол"
B1,"блоггер, маска, страх, надежда, травма, пенальти, алкоголь, классный"
B2,"религиозный, подсчет, новичок, иммунитет, лев, погоня, динамик, густой"
C1,"направленность, выдержка, местоположение, триумф, мандат, проклятый, категорический, стрелковый"
C2,"благословение, пафос, проповедь, выявление, нашествие, величие, раскол, командный"


kelly_ru dict + two CEFR assignment functions.

**Section 6: Candidate Extraction Pipeline**

Filters reduce the initial lemma pool to a curated set of film-review vocabulary candidates.

Pipeline: tokenisation → frequency → stopwords → CEFR → film coverage → Filter A (title artifacts) → Filter B (internationalisms) → frequency normalisation → manual REMOVE → top 20 per language.

**6.1 Tokenisation and POS filtering**

spaCy md models used for accurate lemmatisation. Only NOUN and ADJ retained — primary carriers of evaluative content in film reviews.

In [14]:
nlp_de     = spacy.load('de_core_news_md')
nlp_ru     = spacy.load('ru_core_news_md')
TARGET_POS = {'ADJ','NOUN'}

def extract_lemmas(texts, nlp, batch_size=50):
    counts = Counter()
    for doc in nlp.pipe(texts, batch_size=batch_size,
                        disable=['parser','ner']):
        for token in doc:
            if token.pos_ in TARGET_POS and not token.is_stop:
                lemma = token.lemma_.lower().strip()
                if len(lemma) > 2 and lemma.isalpha():
                    counts[(lemma, token.pos_)] += 1
    return counts

print("Tokenising reviews...")
de_counts = extract_lemmas(df_de['review_de'].dropna().tolist(), nlp_de)
ru_counts = extract_lemmas(df_ru['review_ru'].dropna().tolist(), nlp_ru)
print(f"  DE: {len(de_counts)} types | RU: {len(ru_counts)} types")

Tokenising reviews...
  DE: 3895 types | RU: 8884 types


**6.2 Frequency filter (corpus freq ≥ 5)**

Items below this threshold are statistically marginal.

In [15]:
MIN_FREQ    = 5
de_frequent = {l: c for (l,p),c in de_counts.items() if c >= MIN_FREQ}
ru_frequent = {l: c for (l,p),c in ru_counts.items() if c >= MIN_FREQ}
print(f"\nFreq >= {MIN_FREQ}: {len(de_frequent)} DE | {len(ru_frequent)} RU")


Freq >= 5: 456 DE | 2037 RU


**6.3 Domain stopword removal**

Generic film-discourse words: frequent in any review but carry no genre-specific evaluative content.

In [16]:
GENERIC_DE = {
    'film','filme','kino','geschichte','szene','szenen','figur',
    'charakter','regisseur','schauspieler','rolle','ende','anfang',
    'teil','mann','frau','mensch','welt','leben','zeit','jahr',
    'minute','stunde','name','buch','bild','weise','art',
    'streifen',
}
GENERIC_RU = {
    'фильм','кино','история','сцена','персонаж','герой','режиссёр',
    'актёр','роль','конец','начало','часть','человек','мир','жизнь',
    'время','год','минута','книга','образ','способ','вид','место',
    'вещь','раз','фильма',
}

de_cands = {w: c for w,c in de_frequent.items() if w not in GENERIC_DE}
ru_cands = {w: c for w,c in ru_frequent.items() if w not in GENERIC_RU}
print(f"After stopwords: {len(de_cands)} DE | {len(ru_cands)} RU")

After stopwords: 430 DE | 2012 RU


**6.4 CEFR filter (B1–C1) + anglicism removal**

A1/A2 are too basic for B1+ learners; C2 items are too rare to be productive vocabulary targets. Anglicisms — words where EN Zipf exceeds DE Zipf by more than 0.8 — are likely already known via English and excluded.

In [17]:
def is_anglicism(word, threshold=0.8):
    return (zipf_frequency(word,'en') - zipf_frequency(word,'de')) > threshold

TARGET_CEFR = {'B1','B2','C1'}
rows_cefr   = []
for lemma, freq in de_cands.items():
    if is_anglicism(lemma): continue
    level, source = assign_cefr_de(lemma)
    if level in TARGET_CEFR:
        rows_cefr.append({'lemma':lemma,'lang':'de','corpus_freq':freq,
                          'cefr':level,'cefr_source':source})
for lemma, freq in ru_cands.items():
    level, source = assign_cefr_ru(lemma)
    if level in TARGET_CEFR:
        rows_cefr.append({'lemma':lemma,'lang':'ru','corpus_freq':freq,
                          'cefr':level,'cefr_source':source})

df_cefr = pd.DataFrame(rows_cefr)
print(f"\nAfter CEFR B1-C1: {(df_cefr['lang']=='de').sum()} DE | "
      f"{(df_cefr['lang']=='ru').sum()} RU")
print("\nCEFR source breakdown:")
print(df_cefr.groupby(['lang','cefr_source']).size().to_string())


After CEFR B1-C1: 233 DE | 1201 RU

CEFR source breakdown:
lang  cefr_source 
de    GermanFluent     197
      wordfreq          36
ru    Kelly           1097
      wordfreq         104


**6.5 Film coverage filter (≥ 6 of 15 films)**

A word must appear in reviews of at least MIN_FILMS = 6 films. Words concentrated in one film's reviews are topic artifacts (e.g. тюрьма from Shawshank), not genre-level vocabulary.

In [18]:
MIN_FILMS = 6

def film_coverage(df, text_col, film_col, nlp, candidates):
    cand_set = set(candidates)
    coverage = defaultdict(set)
    for doc, film in zip(
        nlp.pipe(df[text_col].fillna('').tolist(),
                 batch_size=50, disable=['parser','ner']),
        df[film_col].tolist()
    ):
        for token in doc:
            lemma = token.lemma_.lower().strip()
            if lemma in cand_set: coverage[lemma].add(film)
    return {lemma: len(films) for lemma,films in coverage.items()}

print("\nComputing film coverage...")
de_cov = film_coverage(df_de,'review_de','name_en',nlp_de,
                        df_cefr[df_cefr['lang']=='de']['lemma'].tolist())
ru_cov = film_coverage(df_ru,'review_ru','name_en',nlp_ru,
                        df_cefr[df_cefr['lang']=='ru']['lemma'].tolist())

df_cefr['film_count'] = df_cefr.apply(
    lambda r: (de_cov if r['lang']=='de' else ru_cov).get(r['lemma'],0), axis=1)
df_coverage = df_cefr[df_cefr['film_count'] >= MIN_FILMS].copy()
print(f"Film coverage >= {MIN_FILMS}/{N_FILMS}: "
      f"{(df_coverage['lang']=='de').sum()} DE | "
      f"{(df_coverage['lang']=='ru').sum()} RU")


Computing film coverage...
Film coverage >= 6/15: 107 DE | 724 RU


**6.6 Filter A: Title artifact detection (TF-IDF)**

TF-IDF computed across film subcorpora (one document per film). A word is a title artifact if its max(TF-IDF) / mean(TF-IDF) ratio exceeds ARTIFACT_RATIO = 8.0, indicating strong concentration in one film's reviews.

In [19]:
ARTIFACT_RATIO = 8.0

def get_title_artifacts(df, text_col, film_col, ratio=ARTIFACT_RATIO):
    film_texts = df.groupby(film_col)[text_col].apply(' '.join).reset_index()
    vec   = TfidfVectorizer(min_df=2, token_pattern=r'(?u)\b\w{3,}\b')
    tfidf = vec.fit_transform(film_texts[text_col]).toarray()
    names = vec.get_feature_names_out()
    arts  = set()
    for i, word in enumerate(names):
        scores = tfidf[:, i]
        mean   = scores.mean()
        if mean > 0 and scores.max() / mean > ratio:
            arts.add(word)
    return arts

print("\nFilter A: title artifacts...")
de_artifacts = get_title_artifacts(df_de, 'review_de', 'name_en')
ru_artifacts = get_title_artifacts(df_ru, 'review_ru', 'name_en')
df_coverage['is_artifact'] = df_coverage.apply(
    lambda r: r['lemma'] in (de_artifacts if r['lang']=='de'
                              else ru_artifacts), axis=1)
print(f"  DE: {df_coverage[df_coverage['lang']=='de']['is_artifact'].sum()} artifacts")
print(f"  RU: {df_coverage[df_coverage['lang']=='ru']['is_artifact'].sum()} artifacts")


Filter A: title artifacts...
  DE: 8 artifacts
  RU: 80 artifacts


**6.7 Filter B: Transparent internationalisms (DE only)**

A German word is transparent if its normalised form (umlauts replaced) is orthographically similar to English (SequenceMatcher ratio > 0.82) and that English word is frequent (Zipf > 3.0). Such words are likely already known to learners via English.

In [21]:
SIM_THRESHOLD = 0.82
EN_FREQ_MIN   = 3.0

def is_transparent_intl(lemma):
    """Returns (is_intl, similarity, en_zipf, normalised_form)."""
    norm = (lemma.replace('ä','a').replace('ö','o')
                 .replace('ü','u').replace('ß','ss'))
    en_freq = zipf_frequency(norm, 'en')
    if en_freq < EN_FREQ_MIN:
        return False, 0.0, en_freq, norm
    ratio = 1.0 if norm == lemma else \
            difflib.SequenceMatcher(None, norm, lemma).ratio()
    return ratio >= SIM_THRESHOLD, ratio, en_freq, norm

# apply and collect details
intl_records = []
for _, row in df_coverage[df_coverage['lang']=='de'].iterrows():
    is_intl, sim, en_zipf, norm = is_transparent_intl(row['lemma'])
    if is_intl:
        intl_records.append({
            'lemma':     row['lemma'],
            'normalised': norm,
            'similarity': round(sim, 3),
            'en_zipf':   round(en_zipf, 2),
            'cefr':      row.get('cefr', ''),
            'freq_norm': row.get('freq_norm', 0),
            'film_count': row.get('film_count', 0),
        })

df_intl = pd.DataFrame(intl_records).sort_values('en_zipf', ascending=False)

print(f"\nFilter B: {len(df_intl)} DE transparent internationalisms")
print("(thresholds: similarity > {SIM_THRESHOLD}, EN Zipf > {EN_FREQ_MIN})\n")
if not df_intl.empty:
    print(df_intl.to_string(index=False))

# mark on df_coverage as before
df_coverage['is_intl'] = df_coverage.apply(
    lambda r: r['lang']=='de' and is_transparent_intl(r['lemma'])[0], axis=1)


Filter B: 19 DE transparent internationalisms
(thresholds: similarity > {SIM_THRESHOLD}, EN Zipf > {EN_FREQ_MIN})

      lemma  normalised  similarity  en_zipf cefr  freq_norm  film_count
       form        form         1.0     5.31   B1          0           6
     moment      moment         1.0     5.24   B1          0          10
       fall        fall         1.0     5.13   B1          0          15
      drama       drama         1.0     4.67   B2          0          10
  emotional   emotional         1.0     4.66   B2          0          11
     detail      detail         1.0     4.57   B1          0           8
    element     element         1.0     4.46   B2          0           8
      oscar       oscar         1.0     4.31   B2          0           9
  highlight   highlight         1.0     4.21   C1          0           6
    emotion     emotion         1.0     4.19   B2          0           7
      humor       humor         1.0     4.19   B2          0           7
      ge

**6.8 Frequency normalisation via regex**

spaCy lemmatisation and regex matching diverge slightly. Regex-based counts are used as canonical frequencies. Language-specific patterns handle Russian inflection and German capitalisation. The RU_ROOTS dictionary maps problematic lemmas to explicit regex roots (e.g. быт → \bбыть\b-aware patterns) to avoid false positives.

In [22]:
def get_pattern(lemma, lang):
    RU_ROOTS = {
        'сюжет':        r'сюжет\w*',
        'атмосфера':    r'атмосфер\w*',
        'надежда':      r'надежд\w*',
        'судьба':       r'судьб\w*',
        'повествование':r'повествован\w*',
        'смысл':        r'смысл\w*',
        'идея':         r'иде[яию]\w*|идей',
        'итог':         r'\bитог\w*',
        'шедевр':       r'шедевр\w*',
        'сценарий':     r'сценари\w*',
        'впечатление':  r'впечатлен\w*',
        'произведение': r'произведен\w*',
        'финал':        r'\bфинал\w*',
        'драма':        r'драм\w*',
        'визуальный':   r'визуальн\w*',
        'взгляд':       r'\bвзгляд\w*',
        'просмотр':     r'\bпросмотр\w*',
        'кадр':         r'\bкадр\w*',
        'реальность':   r'реальност\w*',
        'уровень':      r'\bуровен\w*|\bуровн\w*',
        'человеческий': r'человеческ\w*',
        'отношение':    r'отношен\w*',
        'реальный':     r'реальн\w*',
        'живой':        r'\bжив(ой|ого|ому|ым|ом|ая|ой|ую|ые|ых|ыми)\b',
        'интерес':      r'\bинтерес(ен|на|но|ны|ного|ному|ным|ном|нее)?\b',
        'тон':          r'\bтон[аеуы]?\b',
        'зло':          r'\bзл[оауе]\b|\bзлод\w+',
        'век':          r'\bвек[аеуов]?\b',
    }
    if lang == 'ru':
        return RU_ROOTS.get(lemma, r'\b' + re.escape(lemma) + r'\w*')
    return r'\b' + lemma[0].upper() + lemma[1:] + r'\w*'

def count_regex(df, text_col, pattern):
    return int(df[text_col].str.count(pattern, flags=re.IGNORECASE).sum())

print("\nRecomputing frequencies via regex...")
df_coverage['regex_freq'] = df_coverage.apply(
    lambda r: count_regex(
        df_de if r['lang']=='de' else df_ru,
        'review_de' if r['lang']=='de' else 'review_ru',
        get_pattern(r['lemma'], r['lang'])
    ), axis=1)
df_coverage['freq_norm'] = df_coverage.apply(
    lambda r: norm_freq(r['regex_freq'],
                        total_tokens_de if r['lang']=='de'
                        else total_tokens_ru), axis=1)

print("\nTop candidates after regex recount:")
top = df_coverage.sort_values(['lang','freq_norm'], ascending=[True,False])
for lang in ['de','ru']:
    print(f"\n  {lang.upper()}:")
    for _, r in top[top['lang']==lang].head(20).iterrows():
        print(f"    {r['lemma']:<20} {r['regex_freq']:>5}  {r['freq_norm']:.2f}/1000t")


Recomputing frequencies via regex...

Top candidates after regex recount:

  DE:
    gute                   118  2.10/1000t
    meister                117  2.09/1000t
    perfekt                104  1.85/1000t
    genial                  91  1.62/1000t
    meisterwerk             90  1.60/1000t
    absolut                 86  1.53/1000t
    emotion                 68  1.21/1000t
    zuschauer               68  1.21/1000t
    fazit                   67  1.19/1000t
    dialog                  65  1.16/1000t
    drama                   53  0.94/1000t
    darsteller              53  0.94/1000t
    oscar                   53  0.94/1000t
    spannung                53  0.94/1000t
    effekt                  51  0.91/1000t
    handlung                50  0.89/1000t
    emotional               50  0.89/1000t
    charaktere              48  0.86/1000t
    genre                   45  0.80/1000t
    voll                    43  0.77/1000t

  RU:
    сюжет                  483  2.65/1000t
    быт 

**6.9 Manual REMOVE + final selection**

Three categories of items removed:

*   (a) False positives from regex matching (быт → быть, псих → психологический)
*   (b) Too general / not genre-specific (момент, ситуация)
*   (c) Intensifiers / function words (absolut, voll, gespielt)

All removals documented with rationale for reproducibility. Final selection: top 20 per language by regex frequency.

In [23]:
MIN_FREQ_NORM = 0.35

REMOVE = {
    # DE: too general / intensifiers / not genre-specific
    'werk', 'grund', 'länge', 'zuschauer',
    'meister',     # regex catches "meisterhafte" — only meisterwerk needed
    'gute',        # lemma "gut", A2 — too basic
    'voll',        # catches "voller" — intensifier
    'absolut',     # intensifier
    'gesamt',      # catches "insgesamt"
    'gespielt',    # participle, not noun
    'effekt',      # transparent internationalism
    'charaktere',  # too general
    'schnitt',     # narrowly technical (editing)
    'anschauen',   # verb
    'passend',     # too general
    'unglaublich', # intensifier
    'stil',        # regex catches "Stille" (silence), not "Stil" (style)
    # RU: regex false positives
    'ход',    # caught "происходит", "выходит", "подходит"
    'быт',    # caught "быть"
    'след',   # caught "следующий", "следить"
    'кое',    # conjunction particle
    'рад',    # caught "ради"
    'поп',    # caught "попытка", "попадает"
    'ряд',    # caught "рядом"
    'разу',   # form of "раз"
    'псих',   # caught "психологический", "психику"
    'благо',  # caught "благодаря"
    # RU: duplicate / wrong spaCy lemmatisation
    'реальный',  # overlaps with "реальность"
    'идеал',     # spaCy lemmatises "идеальный" → "идеал"
    # RU: too general / not genre-specific
    'первый', 'второй', 'третий',
    'момент', 'ситуация', 'уровень', 'отношение',
    'взгляд', 'просмотр', 'событие',
    'живой', 'интерес', 'тон', 'зло', 'век',
}

df_filtered = df_coverage[
    (~df_coverage['is_artifact']) &
    (~df_coverage['is_intl']) &
    (df_coverage['freq_norm'] >= MIN_FREQ_NORM) &
    (~df_coverage['lemma'].isin(REMOVE))
].copy()

print(f"\nAfter all filters: "
      f"{(df_filtered['lang']=='de').sum()} DE | "
      f"{(df_filtered['lang']=='ru').sum()} RU")

df_final = (df_filtered
            .sort_values(['lang','regex_freq'], ascending=[True,False])
            .groupby('lang')
            .head(20)
            .reset_index(drop=True))

print(f"\nFinal dictionary: {len(df_final)} lemmas "
      f"({(df_final['lang']=='de').sum()} DE, "
      f"{(df_final['lang']=='ru').sum()} RU)\n")
print(df_final[['lemma','lang','cefr','cefr_source',
                'regex_freq','freq_norm','film_count']].to_string(index=False))

print(f"\nCEFR distribution:")
print(df_final.groupby(['lang','cefr']).size().to_string())

FINAL_DE = df_final[df_final['lang']=='de']['lemma'].tolist()
FINAL_RU = df_final[df_final['lang']=='ru']['lemma'].tolist()
print(f"\nDE ({len(FINAL_DE)}): {FINAL_DE}")
print(f"RU ({len(FINAL_RU)}): {FINAL_RU}")


After all filters: 18 DE | 34 RU

Final dictionary: 38 lemmas (18 DE, 20 RU)

          lemma lang cefr  cefr_source  regex_freq  freq_norm  film_count
        perfekt   de   B1 GermanFluent         104       1.85          15
         genial   de   B2 GermanFluent          91       1.62          13
    meisterwerk   de   C1 GermanFluent          90       1.60          15
          fazit   de   B2 GermanFluent          67       1.19          15
     darsteller   de   C1 GermanFluent          53       0.94          13
       spannung   de   B2 GermanFluent          53       0.94          11
       handlung   de   C1 GermanFluent          50       0.89          14
         gefühl   de   B1 GermanFluent          41       0.73           8
       gelungen   de   B2     wordfreq          39       0.70           6
       drehbuch   de   B2 GermanFluent          37       0.66          13
   hervorragend   de   B2 GermanFluent          34       0.61          13
     atmosphäre   de   B2 GermanF

**Section 7: Sentiment Lexicon Cross-Validation**

Cross-validation against:

*   SentiWS v2.0 (Remus et al., 2010) — German, ~34k entries
*   RuSentiLex 2017 (Loukachevitch & Levchik, 2016) — Russian, ~13k entries

Both encode direct polarity (positive/negative/neutral) from general news/web corpora. Low coverage of CineLexi candidates confirms that genre-specific evaluative vocabulary is absent from general sentiment resources.

RuSentiLex structure: word, POS, lemma, sentiment, source. Sentiment is in column 3, not 2 — the parser searches all columns for a sentiment label. This handles both rusentilex_2017.txt (tab-separated) and possible comma-separated variants.

In [24]:
def load_sentiws(content_bytes):
    result = {}
    with zipfile.ZipFile(io.BytesIO(content_bytes)) as z:
        for fname in z.namelist():
            if not fname.endswith('.txt'): continue
            polarity = 'positive' if 'Positive' in fname else 'negative'
            with z.open(fname) as f:
                for line in f.read().decode('utf-8').splitlines():
                    line = line.strip()
                    if not line or line.startswith('#'): continue
                    parts = line.split('\t')
                    if len(parts) < 2: continue
                    lemma = parts[0].split('|')[0].lower().strip()
                    result[lemma] = polarity
                    if len(parts) > 2:
                        for form in parts[2].split(','):
                            fc = form.strip().lower()
                            if fc: result[fc] = polarity
    return result

def parse_rusentilex(text):
    result = {}
    sentiments = {'positive','negative','neutral','both'}
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith('!'): continue
        sep   = '\t' if '\t' in line else ','
        parts = [p.strip() for p in line.split(sep)]
        if len(parts) < 3: continue
        lemma = parts[0].lower()
        sentiment = next((p.lower() for p in parts[1:]
                          if p.lower() in sentiments), None)
        if sentiment and lemma:
            if lemma not in result:
                result[lemma] = sentiment
    return result

sentiws = {}
print("Loading SentiWS...")
for url in ['https://downloads.wortschatz-leipzig.de/etc/SentiWS/SentiWS_v2.0.zip',
            'http://pcai056.informatik.uni-leipzig.de/downloads/etc/SentiWS/SentiWS_v2.0.zip']:
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200 and r.content[:2] == b'PK':
            sentiws = load_sentiws(r.content)
            print(f"  Loaded: {len(sentiws)} entries"); break
    except Exception as e: print(f"  {e}")
if not sentiws:
    print("  Manual upload required.")
    up_s = files.upload()
    for fname, data in up_s.items():
        if fname.endswith('.zip'):
            sentiws = load_sentiws(data)
            print(f"  Loaded: {len(sentiws)} entries")

rusenti = {}
print("Loading RuSentiLex...")
for url in ['http://www.labinform.ru/pub/rusentilex/rusentilex_2017.txt']:
    try:
        r = requests.get(url, timeout=30)
        if r.status_code == 200 and len(r.content) > 10000:
            text = r.content.decode('utf-8')
            rusenti = parse_rusentilex(text)
            if len(rusenti) > 100:
                print(f"  Loaded: {len(rusenti)} entries"); break
    except Exception as e: print(f"  {e}")
if len(rusenti) < 100:
    print("  Automatic download failed — upload rusentilex_2017.txt manually")
    print("  Download from: http://www.labinform.ru/pub/rusentilex/")
    up_r = files.upload()
    for fname, data in up_r.items():
        text = data.decode('utf-8')
        rusenti = parse_rusentilex(text)
        print(f"  Loaded: {len(rusenti)} entries")

de_in     = [(w, sentiws[w])  for w in FINAL_DE if w in sentiws]
ru_in     = [(w, rusenti[w]) for w in FINAL_RU if w in rusenti]
total_cov = len(de_in) + len(ru_in)
n_total   = len(FINAL_DE) + len(FINAL_RU)

print(f"\n Sentiment Lexicon Coverage")
print(f"\nDE — SentiWS ({len(de_in)}/{len(FINAL_DE)}, "
      f"{100*len(de_in)//max(len(FINAL_DE),1)}%):")
for w,p in de_in: print(f"  + {w}: {p}")

print(f"\nRU — RuSentiLex ({len(ru_in)}/{len(FINAL_RU)}, "
      f"{100*len(ru_in)//max(len(FINAL_RU),1)}%):")
for w,p in ru_in: print(f"  + {w}: {p}")
for w in FINAL_RU:
    if w not in rusenti: print(f"  - {w}: absent")

print(f"\nOverall: {total_cov}/{n_total} "
      f"({100*total_cov//n_total if n_total else 0}%)")

Loading SentiWS...
  Loaded: 34136 entries
Loading RuSentiLex...
  Loaded: 13326 entries

 Sentiment Lexicon Coverage

DE — SentiWS (6/18, 33%):
  + perfekt: positive
  + genial: positive
  + meisterwerk: positive
  + gelungen: positive
  + hervorragend: positive
  + düster: negative

RU — RuSentiLex (3/20, 15%):
  + реальность: neutral
  + человеческий: neutral
  + шедевр: positive
  - сюжет: absent
  - смысл: absent
  - драма: absent
  - атмосфера: absent
  - идея: absent
  - кадр: absent
  - сценарий: absent
  - роман: absent
  - финал: absent
  - визуальный: absent
  - итог: absent
  - кинематограф: absent
  - состав: absent
  - повествование: absent
  - жанр: absent
  - произведение: absent
  - судьба: absent

Overall: 9/38 (23%)


SentiWS and RuSentiLex cover only ~23% of CineLexi candidates. Items absent from both — narrative structure terms, craft vocabulary, discourse markers — represent the genre-specific evaluative vocabulary that CineLexi uniquely contributes.

**Section 8: Dictionary Entry Generation**

For each of the final lemmas:

*   (a) Normalised frequency per 1,000 tokens (regex-based)
*   (b) Two corpus examples from different films (shortest sentence ≤ 300 chars preferred)
*   (c) Up to 3 collocations — content word most frequently immediately preceding the lemma
*   (d) Up to 5 chunks — frequent 2–3 word sequences containing the lemma (production-ready phrases)

In [31]:
def find_sentences(text, pattern):
    if pd.isna(text): return []
    # Normalise line breaks and spacing
    text = re.sub(r'\s+', ' ', str(text))
    # Insert space between sentence-final punctuation and capitalised word
    text = re.sub(r'([.!?])([A-ZÄÖÜА-ЯЁ])', r'\1 \2', text)
    # Split on sentence boundaries
    sents = re.split(r'(?<=[.!?])\s+(?=[A-ZÄÖÜА-ЯЁ])', text)
    results = []
    for s in sents:
        s = s.strip()
        if not re.search(pattern, s, re.IGNORECASE): continue
        if len(s) < 20: continue
        if s[-1] not in '.!?': continue
        # Reject truncated endings (z.B., usw., etc.)
        if re.search(r'\b(z\.B\.|usw\.|etc\.|d\.h\.|u\.a\.)\s*$', s): continue
        if len(s) > 250: continue
        results.append(s)
    return results

def get_examples(df, text_col, film_col, year_col, pattern, n=2):
    matches, seen = [], set()
    for _, row in df.iterrows():
        if row[film_col] in seen: continue
        sents = find_sentences(row[text_col], pattern)
        if sents:
            best = min(sents, key=len)
            matches.append({'film': row[film_col],
                            'year': row[year_col],
                            'sentence': best})
            seen.add(row[film_col])
        if len(matches) >= n: break
    return matches

def get_collocations(df, text_col, pattern, nlp, top_n=3):
    STOP = {
        'ein','eine','einen','einem','einer','der','die','das',
        'des','den','dem','und','oder','ist','war','hat','mit',
        'von','für','auf','in','an','zu','sich','auch','nicht',
        'viel','mehr','sehr','nur','noch','immer','schon',
        'dieser','diese','dieses','diesem','diesen',
        'mein','meine','meinen','meinem','meiner',
        'sein','seine','seinen','seinem','seiner',
        'ihr','ihre','ihren','ihrem','ihrer',
        'kann','können','muss','müssen','will','wollen',
        'wird','werden','wurde','wurden','haben','hatte',
        'durch','über','unter','zwischen','gegen','ohne',
        'als','wie','wenn','dass','weil','aber','doch',
        'не','это','в','на','с','и','а','но','как','что','по',
        'из','за','он','она','они','уже','ещё','так','где',
        'очень','тоже','только','даже','для','при','под','над',
        'между','через','без','про','о','об','от','до','к',
        'его','её','их','свой','своя','своё','свои',
        'есть','имеет','может','будет','был','была','были',
        'этот','эта','это','эти','тот','та','те',
        'весь','вся','всё','все','сам','сама','само','сами',
    }
    before = Counter()
    for text in df[text_col].dropna():
        doc = nlp(text)
        tokens = list(doc)
        for i, tok in enumerate(tokens):
            if re.search(pattern, tok.text, re.IGNORECASE) and i > 0:
                prev = tokens[i-1]
                if prev.pos_ not in ('ADJ', 'NOUN'): continue
                if prev.is_stop: continue
                lemma_prev = prev.lemma_.lower().strip()
                if lemma_prev in STOP: continue
                if len(lemma_prev) < 4: continue
                before[lemma_prev] += 1
    return [w for w,_ in before.most_common(top_n)]

def get_chunks(df, text_col, pattern, lang, nlp, top_n=5, min_count=3):
    BAD_START_DE = {'der','die','das','des','dem','den',
                    'ein','eine','einen','einem','einer',
                    'und','oder','aber','denn','weil','dass',
                    'in','an','auf','mit','von','zu','für','bei','nach'}
    BAD_END_DE   = {'in','an','auf','mit','von','zu','für','bei','nach',
                    'und','oder','aber','denn','weil','dass',
                    'ein','eine','dem','den'}
    BAD_START_RU = {'в','на','с','и','а','но','к','по','из','за','от',
                    'до','для','при','под','над','через','без','про'}
    BAD_END_RU   = {'в','на','с','и','а','но','к','по','из','за','от',
                    'до','для','при','под','над','через','без','про',
                    'что','чтобы','как','это','его','её','их'}

    def clean(chunk):
        toks = chunk.split()
        if len(toks) < 2: return False
        if lang == 'de':
            if toks[0] in BAD_START_DE: return False
            if toks[-1] in BAD_END_DE: return False
        else:
            if toks[0] in BAD_START_RU: return False
            if toks[-1] in BAD_END_RU: return False
        # Must contain at least one content word besides the lemma
        doc = nlp(chunk)
        content = [t for t in doc if t.pos_ in ('ADJ', 'NOUN')
                   and not t.is_stop
                   and not re.search(pattern, t.text, re.IGNORECASE)]
        return len(content) >= 1

    chunk_counter = Counter()
    for text in df[text_col].dropna():
        tokens = text.split()
        for i, tok in enumerate(tokens):
            if not re.search(pattern, tok, re.IGNORECASE): continue
            for size in [2, 3]:
                for start in range(max(0, i-size+1),
                                   min(i+1, len(tokens)-size+1)):
                    chunk_toks = tokens[start:start+size]
                    if not any(re.search(pattern, t, re.IGNORECASE)
                               for t in chunk_toks): continue
                    chunk = ' '.join(
                        re.sub(r'[^\wäöüÄÖÜßа-яА-ЯёЁ\-]','', t).lower()
                        for t in chunk_toks
                    ).strip()
                    if chunk and len(chunk) > 3:
                        chunk_counter[chunk] += 1
    return [ch for ch,cnt in chunk_counter.most_common(top_n*5)
            if cnt >= min_count and clean(ch)][:top_n]

# Lemmas that do not form productive chunks (metadiscursive markers)
NO_CHUNKS = {'fazit', 'итог'}

entries = []
print("Generating dictionary entries...\n")

for _, row in df_final.iterrows():
    lemma, lang = row['lemma'], row['lang']
    pattern = get_pattern(lemma, lang)
    if lang == 'de':
        df_src, text_col, total, nlp = df_de, 'review_de', total_tokens_de, nlp_de
    else:
        df_src, text_col, total, nlp = df_ru, 'review_ru', total_tokens_ru, nlp_ru

    count    = count_regex(df_src, text_col, pattern)
    examples = get_examples(df_src, text_col, 'name_en', 'year', pattern)
    collocs  = get_collocations(df_src, text_col, pattern, nlp)
    chunks   = [] if lemma in NO_CHUNKS else \
               get_chunks(df_src, text_col, pattern, lang, nlp)

    entry = {
        'lemma': lemma, 'lang': lang,
        'cefr': row['cefr'], 'cefr_source': row['cefr_source'],
        'corpus_freq': count, 'freq_per_1000': norm_freq(count, total),
        'film_count': row['film_count'],
    }
    for i, ex in enumerate(examples[:2], start=1):
        entry[f'ex_{i}_film']     = ex['film']
        entry[f'ex_{i}_year']     = ex['year']
        entry[f'ex_{i}_sentence'] = ex['sentence']
    for i, c in enumerate(collocs[:3], start=1):
        entry[f'colloc_{i}'] = c
    for i, ch in enumerate(chunks[:5], start=1):
        entry[f'chunk_{i}'] = ch
    entries.append(entry)
    print(f"  [{lang.upper()}] {lemma}: {count} ({norm_freq(count,total)}/1000t)"
          f" | {len(examples)} ex | {len(collocs)} collocs | {len(chunks)} chunks")

df_entries = pd.DataFrame(entries)
df_entries.to_csv('cinelexi_entries.csv', index=False, encoding='utf-8-sig')
print(f"\nSaved: cinelexi_entries.csv ({len(df_entries)} entries)")

Generating dictionary entries...

  [DE] perfekt: 104 (1.85/1000t) | 2 ex | 3 collocs | 3 chunks
  [DE] genial: 91 (1.62/1000t) | 2 ex | 3 collocs | 2 chunks
  [DE] meisterwerk: 90 (1.6/1000t) | 2 ex | 3 collocs | 3 chunks
  [DE] fazit: 67 (1.19/1000t) | 2 ex | 1 collocs | 0 chunks
  [DE] darsteller: 53 (0.94/1000t) | 2 ex | 3 collocs | 1 chunks
  [DE] spannung: 53 (0.94/1000t) | 2 ex | 3 collocs | 0 chunks
  [DE] handlung: 50 (0.89/1000t) | 2 ex | 3 collocs | 0 chunks
  [DE] gefühl: 41 (0.73/1000t) | 2 ex | 3 collocs | 0 chunks
  [DE] gelungen: 39 (0.7/1000t) | 2 ex | 3 collocs | 0 chunks
  [DE] drehbuch: 37 (0.66/1000t) | 2 ex | 3 collocs | 0 chunks
  [DE] hervorragend: 34 (0.61/1000t) | 2 ex | 1 collocs | 0 chunks
  [DE] atmosphäre: 32 (0.57/1000t) | 2 ex | 3 collocs | 1 chunks
  [DE] hauptdarsteller: 31 (0.55/1000t) | 2 ex | 3 collocs | 0 chunks
  [DE] düster: 26 (0.46/1000t) | 2 ex | 0 collocs | 1 chunks
  [DE] leinwand: 22 (0.39/1000t) | 2 ex | 0 collocs | 0 chunks
  [DE] tiefe: 

In [38]:
def print_entry(row, max_examples=2, max_collocs=3, max_chunks=5):
    """Pretty-print one dictionary entry."""
    print(f"  [{row['lang'].upper()}] {row['lemma']}  "
          f"(CEFR {row['cefr']}, {row['cefr_source']})")
    print(f"  freq: {row['corpus_freq']} ({row['freq_per_1000']}/1000t) | "
          f"films: {row['film_count']}/15")

    # examples
    has_ex = False
    for i in [1, 2]:
        film = row.get(f'ex_{i}_film', '')
        year = row.get(f'ex_{i}_year', '')
        sent = row.get(f'ex_{i}_sentence', '')
        if pd.isna(sent) or not str(sent).strip():
            continue
        if not has_ex:
            print("  Examples:")
            has_ex = True
        s = str(sent)
        s_display = s if len(s) <= 200 else s[:200] + '...'
        print(f"    {i}. [{film}, {year}]")
        print(f"       {s_display}")

    # collocations
    collocs = [str(row.get(f'colloc_{i}', '')).strip()
               for i in range(1, max_collocs+1)]
    collocs = [c for c in collocs if c and c != 'nan']
    if collocs:
        print(f"  Collocations: {' | '.join(collocs)}")

    # chunks
    chunks = [str(row.get(f'chunk_{i}', '')).strip()
              for i in range(1, max_chunks+1)]
    chunks = [c for c in chunks if c and c != 'nan']
    if chunks:
        print(f"  Chunks: {' | '.join(chunks)}")


# print all entries grouped by language
for lang in ['de', 'ru']:
    subset = df_entries[df_entries['lang'] == lang]
    print(f" {lang.upper()} ENTRIES ({len(subset)} total)")
    for _, row in subset.iterrows():
        print_entry(row)

 DE ENTRIES (18 total)
  [DE] perfekt  (CEFR B1, GermanFluent)
  freq: 104 (1.85/1000t) | films: 15/15
  Examples:
    1. [The Shawshank Redemption, 1994]
       Besetzung ist nahe zu perfekt ausgewählt.
    2. [The Green Mile, 1999]
       Die Schauspieler haben ihre Rollen perfekt gespielt.
  Collocations: rolle | film | szene
  Chunks: perfekte mischung | perfekte mischung aus | perfekter film
  [DE] genial  (CEFR B2, GermanFluent)
  freq: 91 (1.62/1000t) | films: 13/15
  Examples:
    1. [The Shawshank Redemption, 1994]
       Manchmal bedarf es einfach nicht vieler Worte: dies ist mit absoluter Sicherheit einer der besten Filme aller Zeiten, ein genial gespieltes und Inszeniertes Drama voller Anspruch, Emotionen – und selt...
    2. [The Green Mile, 1999]
       Die Schauspieler harmonieren wunderbar und die Story ist genial.
  Collocations: film | geschichte | bild
  Chunks: geniale dialoge | genialen soundtrack
  [DE] meisterwerk  (CEFR C1, GermanFluent)
  freq: 90 (1.6/1000t) |

**Section 9: XML Dictionary Generation**

Two-block XML structure:

*   entries — monolingual entries (DE + RU). Each: headword, POS, CEFR (with source), frequency, examples, collocations, chunks
*   crosslingual — 6 typological evaluative categories. Each groups lemmas expressing the same evaluative dimension in both languages, with a note on cross-lingual asymmetries

CineLexi is a cross-lingually annotated monolingual resource, not a translation dictionary. The crosslingual block makes the core finding — DE evaluates craft, RU evaluates meaning — explicit and pedagogically accessible.

In [32]:
CATEGORIES = [
    {'id': 'supreme_quality', 'label': 'Supreme quality',
     'de': ['meisterwerk'], 'ru': ['шедевр'],
     'note': ('Prototypical quality nouns: German Meisterwerk '
              '(compound: Meister+Werk) and Russian шедевр '
              '(from French chef-d\'oeuvre). Both are the highest-'
              'frequency quality markers in their respective subcorpora.')},
    {'id': 'narrative_structure', 'label': 'Narrative structure',
     'de': ['handlung','drehbuch'], 'ru': ['сюжет','повествование','сценарий'],
     'note': ('German evaluates narrative through Handlung (plot as '
              'experienced) and Drehbuch (script as craft object). '
              'Russian distributes across сюжет (plot/story), '
              'повествование (mode of narration), and сценарий '
              '(screenplay). The three-way Russian distinction has '
              'no direct German parallel.')},
    {'id': 'emotional_atmospheric', 'label': 'Emotional and atmospheric quality',
     'de': ['atmosphäre','gefühl','spannung'], 'ru': ['атмосфера'],
     'note': ('German has three distinct emotional-atmospheric nouns: '
              'Atmosphäre (ambient mood), Gefühl (felt sense), and '
              'Spannung (tension arc). Russian concentrates this '
              'evaluative dimension primarily in атмосфера.')},
    {'id': 'performance_craft', 'label': 'Performance and craft',
     'de': ['darsteller','hauptdarsteller','drehbuch'], 'ru': [],
     'note': ('German reviews explicitly evaluate acting through '
              'Darsteller and Hauptdarsteller (lead actor), and '
              'script quality through Drehbuch. No equivalent '
              'high-frequency craft vocabulary appears in the Russian '
              'subcorpus, which tends to evaluate thematic content '
              'over technical execution.')},
    {'id': 'meaning_depth', 'label': 'Meaning and thematic depth',
     'de': [], 'ru': ['смысл','идея','судьба'],
     'note': ('Russian reviewers evaluate films through existential '
              'and philosophical categories: смысл (meaning/sense), '
              'идея (idea/concept), судьба (fate/destiny). No German '
              'counterparts appear at comparable frequency, reflecting '
              'different evaluative conventions across the two platforms.')},
    {'id': 'discourse_markers', 'label': 'Review discourse markers',
     'de': ['fazit'], 'ru': ['итог'],
     'note': ('Metadiscursive verdict markers: German Fazit and '
              'Russian итог both frame the reviewer\'s final '
              'evaluation. Genre-structuring rather than content-'
              'evaluating, their high frequency confirms pedagogical '
              'relevance for learners producing film reviews.')},
]

def xe(text):
    if pd.isna(text): return ''
    return (str(text).replace('&','&amp;').replace('<','&lt;')
                     .replace('>','&gt;').replace('"','&quot;'))

def make_xml_entry(row):
    lang, lemma = row['lang'], row['lemma']
    src      = row.get('cefr_source','wordfreq')
    cefr_src = src if src in ('GermanFluent','Goethe','Kelly') else 'wordfreq'
    collocs  = [str(row.get(f'colloc_{i}','')).strip() for i in range(1,4)
                if str(row.get(f'colloc_{i}','')).strip()
                and not pd.isna(row.get(f'colloc_{i}',''))]
    chunks   = [str(row.get(f'chunk_{i}','')).strip() for i in range(1,6)
                if str(row.get(f'chunk_{i}','')).strip()
                and not pd.isna(row.get(f'chunk_{i}',''))]
    lines = [
        f'  <entry lang="{lang}" id="{xe(lemma)}">',
        f'    <headword>{xe(lemma)}</headword>',
        f'    <pos>noun</pos>',
        f'    <cefr source="{cefr_src}">{xe(row["cefr"])}</cefr>',
        f'    <domain>film review</domain>',
        f'    <frequency corpus="CineLexi-{lang.upper()}"'
        f' per_1000="{row.get("freq_per_1000","")}"'
        f' film_count="{row.get("film_count","")}">'
        f'{row.get("corpus_freq","")}</frequency>',
        f'    <examples>',
    ]
    for i in [1,2]:
        film = row.get(f'ex_{i}_film','')
        year = row.get(f'ex_{i}_year','')
        text = row.get(f'ex_{i}_sentence','')
        if pd.isna(text) or not str(text).strip(): continue
        try: yr = int(year)
        except: yr = ''
        lines += [f'      <example film="{xe(str(film))}" year="{yr}">',
                  f'        {xe(str(text).strip())}',
                  f'      </example>']
    lines.append('    </examples>')
    if collocs:
        lines.append('    <collocations>')
        for c in collocs: lines.append(f'      <colloc>{xe(c)}</colloc>')
        lines.append('    </collocations>')
    if chunks:
        lines.append('    <chunks>')
        for ch in chunks: lines.append(f'      <chunk>{xe(ch)}</chunk>')
        lines.append('    </chunks>')
    lines.append('  </entry>')
    return '\n'.join(lines)

n_films = len(df_de['name_en'].unique())
xml = [
    '<?xml version="1.0" encoding="UTF-8"?>',
    '<dictionary id="cinelexi" version="3.0"',
    '            domain="film_review" langs="de ru"',
    '            target_cefr="B1-C1"',
    f'            n_films="{n_films}"',
    f'            genres="drama sci-fi crime/thriller action comedy/romance"',
    f'            corpus_size_de="{total_tokens_de}"',
    f'            corpus_size_ru="{total_tokens_ru}">',
    '',
    '  <!-- MONOLINGUAL ENTRIES: 18 German + 20 Russian -->',
    '  <entries>', '',
    '    <!-- German entries -->',
]

for lemma in FINAL_DE:
    r = df_entries[df_entries['lemma']==lemma]
    if not r.empty:
        xml.append(make_xml_entry(r.iloc[0])); xml.append('')
        print(f"  DE: {lemma}")

xml.append('    <!-- Russian entries -->')
for lemma in FINAL_RU:
    r = df_entries[df_entries['lemma']==lemma]
    if not r.empty:
        xml.append(make_xml_entry(r.iloc[0])); xml.append('')
        print(f"  RU: {lemma}")

xml += ['  </entries>', '',
        '  <!-- CROSSLINGUAL COMPARISON: 6 evaluative categories -->',
        '  <crosslingual>', '']

for cat in CATEGORIES:
    xml.append(f'    <category id="{cat["id"]}"'
               f' label="{xe(cat["label"])}">')
    for code, lemmas in [('de',cat['de']),('ru',cat['ru'])]:
        if lemmas:
            xml.append(f'      <lang code="{code}">')
            for l in lemmas: xml.append(f'        <ref entry="{l}"/>')
            xml.append('      </lang>')
    xml += [f'      <note>{xe(cat["note"])}</note>',
            '    </category>', '']
    print(f"  category: {cat['id']}")

xml += ['  </crosslingual>', '', '</dictionary>']
xml_output = '\n'.join(xml)
with open('cinelexi_dictionary.xml','w',encoding='utf-8') as f:
    f.write(xml_output)
print(f"\nSaved: cinelexi_dictionary.xml "
      f"({xml_output.count('<entry ')} entries, "
      f"{xml_output.count('<category ')} categories)")

  DE: perfekt
  DE: genial
  DE: meisterwerk
  DE: fazit
  DE: darsteller
  DE: spannung
  DE: handlung
  DE: gefühl
  DE: gelungen
  DE: drehbuch
  DE: hervorragend
  DE: atmosphäre
  DE: hauptdarsteller
  DE: düster
  DE: leinwand
  DE: tiefe
  DE: visuell
  DE: regie
  RU: сюжет
  RU: смысл
  RU: драма
  RU: атмосфера
  RU: идея
  RU: кадр
  RU: сценарий
  RU: реальность
  RU: человеческий
  RU: роман
  RU: финал
  RU: шедевр
  RU: визуальный
  RU: итог
  RU: кинематограф
  RU: состав
  RU: повествование
  RU: жанр
  RU: произведение
  RU: судьба
  category: supreme_quality
  category: narrative_structure
  category: emotional_atmospheric
  category: performance_craft
  category: meaning_depth
  category: discourse_markers

Saved: cinelexi_dictionary.xml (38 entries, 6 categories)


**Section 10: Supplementary Candidate List**

Full automatically-filtered list before manual REMOVE. Published as supplementary data on GitHub/Zenodo. Items with known regex false positives (быт, поп, рад, псих, благо, след) are included in the supplementary list but flagged in the paper as requiring manual validation before use.

In [33]:
df_candidates_full = df_coverage[
    (~df_coverage['is_artifact']) &
    (~df_coverage['is_intl']) &
    (df_coverage['freq_norm'] >= MIN_FREQ_NORM)
].copy().sort_values(['lang','regex_freq'], ascending=[True,False]).reset_index(drop=True)

COLS = ['lemma','lang','cefr','cefr_source','regex_freq','freq_norm','film_count']
print(f"\nFull candidate list: "
      f"{(df_candidates_full['lang']=='de').sum()} DE | "
      f"{(df_candidates_full['lang']=='ru').sum()} RU\n")
print(" DE")
print(df_candidates_full[df_candidates_full['lang']=='de'][COLS].to_string(index=False))
print("\n RU")
print(df_candidates_full[df_candidates_full['lang']=='ru'][COLS].to_string(index=False))

with pd.ExcelWriter('cinelexi_candidates_full.xlsx', engine='openpyxl') as writer:
    df_candidates_full[df_candidates_full['lang']=='de'][COLS].to_excel(
        writer, sheet_name='DE', index=False)
    df_candidates_full[df_candidates_full['lang']=='ru'][COLS].to_excel(
        writer, sheet_name='RU', index=False)
    df_candidates_full[COLS].to_excel(writer, sheet_name='ALL', index=False)

print(f"\nSaved: cinelexi_candidates_full.xlsx")
print(f"  DE: {(df_candidates_full['lang']=='de').sum()} lemmas")
print(f"  RU: {(df_candidates_full['lang']=='ru').sum()} lemmas")
print(f"  ALL: {len(df_candidates_full)} lemmas total")


Full candidate list: 34 DE | 51 RU

 DE
          lemma lang cefr  cefr_source  regex_freq  freq_norm  film_count
           gute   de   B2 GermanFluent         118       2.10           8
        meister   de   B1 GermanFluent         117       2.09          12
        perfekt   de   B1 GermanFluent         104       1.85          15
         genial   de   B2 GermanFluent          91       1.62          13
    meisterwerk   de   C1 GermanFluent          90       1.60          15
        absolut   de   B1 GermanFluent          86       1.53          15
      zuschauer   de   B1 GermanFluent          68       1.21          13
          fazit   de   B2 GermanFluent          67       1.19          15
     darsteller   de   C1 GermanFluent          53       0.94          13
       spannung   de   B2 GermanFluent          53       0.94          11
         effekt   de   B2 GermanFluent          51       0.91          10
       handlung   de   C1 GermanFluent          50       0.89          

**Section 11: Summary — canonical figures for the paper**

All figures verified against actual pipeline runs in this notebook. The funnel table (POS → freq → stopwords → CEFR → film coverage → filters → final) is the canonical source for Table 2 in the paper.

In [34]:
print("SUMMARY — canonical figures for the paper")

gf_cov    = (df_final[df_final['lang']=='de']['cefr_source']=='GermanFluent').sum()
kelly_cov = (df_final[df_final['lang']=='ru']['cefr_source']=='Kelly').sum()

funnel = [
    ("POS filter (NOUN + ADJ)",            len(de_counts),              len(ru_counts)),
    ("Frequency >= 5",                     len(de_frequent),            len(ru_frequent)),
    ("Domain stopword removal",            len(de_cands),               len(ru_cands)),
    ("CEFR B1-C1 + anglicism filter",     (df_cefr['lang']=='de').sum(),(df_cefr['lang']=='ru').sum()),
    (f"Film coverage >= {MIN_FILMS}/15",  (df_coverage['lang']=='de').sum(),(df_coverage['lang']=='ru').sum()),
    ("Filter A + B + freq norm + REMOVE", (df_filtered['lang']=='de').sum(),(df_filtered['lang']=='ru').sum()),
    ("Final dictionary",                   len(FINAL_DE),               len(FINAL_RU)),
]

print(f"\n{'Step':<46} {'DE':>5} {'RU':>5}")
for label, de, ru in funnel:
    print(f"{label:<46} {de:>5} {ru:>5}")

print(f"\nCorpus: {n_films} films × 5 genres (3 per genre) × "
      f"{TARGET_PER_FILM} reviews × 2 languages = "
      f"{n_films*TARGET_PER_FILM*2} reviews")
print(f"  DE: {total_tokens_de:,} tokens | avg {df_de['length_de'].mean():.0f} chars/review")
print(f"  RU: {total_tokens_ru:,} tokens | avg {df_ru['length_ru'].mean():.0f} chars/review")
print(f"  Length ratio: {df_ru['length_ru'].mean()/df_de['length_de'].mean():.1f}x "
      f"→ normalisation required")

print(f"\nCEFR sources:")
print(f"  DE: GermanFluent {gf_cov}/{len(FINAL_DE)} "
      f"({100*gf_cov//max(len(FINAL_DE),1)}%) | "
      f"wordfreq {len(FINAL_DE)-gf_cov}/{len(FINAL_DE)}")
print(f"  RU: Kelly {kelly_cov}/{len(FINAL_RU)} "
      f"({100*kelly_cov//max(len(FINAL_RU),1)}%)")

print(f"\nSentiment cross-validation: {total_cov}/{n_total} "
      f"({100*total_cov//n_total if n_total else 0}%)")
print(f"  SentiWS (DE): {len(de_in)}/{len(FINAL_DE)} — "
      f"covers explicit polarity items only")
print(f"  RuSentiLex (RU): {len(ru_in)}/{len(FINAL_RU)} — "
      f"2 tagged 'neutral', 1 'positive'")

print(f"\nFilter thresholds (reported in paper Section 3):")
print(f"  A (TF-IDF artifacts):   max/mean > {ARTIFACT_RATIO}")
print(f"  B (internationalisms):  orthographic sim > {SIM_THRESHOLD}, "
      f"EN Zipf > {EN_FREQ_MIN}")
print(f"  Freq norm:              >= {MIN_FREQ_NORM}/1000 tokens")
print(f"  Film coverage:          >= {MIN_FILMS}/{n_films} films")

print(f"\nSupplementary data: {len(df_candidates_full)} lemmas "
      f"({(df_candidates_full['lang']=='de').sum()} DE + "
      f"{(df_candidates_full['lang']=='ru').sum()} RU)")

files.download('kinopoisk_reviews.csv')
files.download('filmstarts_user_reviews.csv')
files.download('cinelexi_entries.csv')
files.download('cinelexi_dictionary.xml')
files.download('cinelexi_candidates_full.xlsx')
print("\nAll downloads ready.")

SUMMARY — canonical figures for the paper

Step                                              DE    RU
POS filter (NOUN + ADJ)                         3895  8884
Frequency >= 5                                   456  2037
Domain stopword removal                          430  2012
CEFR B1-C1 + anglicism filter                    233  1201
Film coverage >= 6/15                            107   724
Filter A + B + freq norm + REMOVE                 18    34
Final dictionary                                  18    20

Corpus: 15 films × 5 genres (3 per genre) × 30 reviews × 2 languages = 900 reviews
  DE: 56,098 tokens | avg 824 chars/review
  RU: 182,366 tokens | avg 2769 chars/review
  Length ratio: 3.4x → normalisation required

CEFR sources:
  DE: GermanFluent 16/18 (88%) | wordfreq 2/18
  RU: Kelly 20/20 (100%)

Sentiment cross-validation: 9/38 (23%)
  SentiWS (DE): 6/18 — covers explicit polarity items only
  RuSentiLex (RU): 3/20 — 2 tagged 'neutral', 1 'positive'

Filter thresholds (re

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


All downloads ready.
